# SD1 / SD2 (San Diego) — DART + DARTS (+CFAR) + baselines, noise-before
Frozen camera-ready recipe: noise BEFORE the std front (raw isotropic),
DART Adam 5e-4 **wd=0** clip 1.0; DARTS AdamW 3e-4 wd 1e-5 (published
optimizer; `WD` env overrides) clip 1.0; theta=.075; scene signatures from
the protocol (aircraft mean, median-norm scaled). Saves fp16 checkpoints at
every eval, best+final model states, final raw score maps (plain + CFAR),
full pd/auc curves. Restart-safe: finished (det, scene, rho, seed) keys skip.
Edit the CONFIG cell, run top to bottom; zip cell at the end.

In [ ]:
!git clone -b camera-ready --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, torch
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert os.path.exists('repro/data/Sandiego.mat'), 'missing SD1 data'
assert os.path.exists('repro/data/Sandiego2.mat'), 'missing SD2 data'

In [ ]:
# ======================= CONFIG — edit me =======================
SCENES      = 'sandiego,sandiego2'   # sd1, sd2
SEEDS       = '42,43'
THETA       = 0.075
# DART (noise-before, std front, wd=0, clip 1.0, batch 512)
DART_RHOS   = '0.001,0.01,0.1'
DART_EPOCHS = 30000
# DARTS (published AdamW wd 1e-5; set DARTS_WD='0' to ablate)
DARTS_RHOS   = '0.03,0.5'
DARTS_EPOCHS = 10000
DARTS_WD     = '1e-5'
CKPT_EVERY   = 200        # eval + fp16 snapshot cadence (epochs)
# ================================================================
import os
os.environ.update(dict(SCENES=SCENES, SEEDS=SEEDS, THETA=str(THETA),
                       CKPT_EVERY=str(CKPT_EVERY)))

In [ ]:
%%writefile run_sd.py
"""SD1/SD2 engine — DART + DARTS, noise-before, std front (no floors).
Env: DET=dart|darts, SCENES, RHOS, SEEDS, EPOCHS, THETA, CKPT_EVERY,
WD (darts only). Outputs: results_sd.json (curves + best/final),
ckpt_sd/<key>/ (fp16 snaps + model_best/final + scores npz)."""
import json
import os
import sys
import time

sys.path.insert(0, os.getcwd())

import numpy as np
import torch
from tqdm import tqdm

from repro import scenes
from repro.protocols.spatial import load_cfg
from repro.core.data import Whitening, plant_targets, extract_neighborhoods
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.core.seeding import seed_all
from repro.core.models import ScoreNet
from repro.models.darts.model import DARTS, _NeighborDenoiser

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_grad_enabled(True)
DET = os.environ.get('DET', 'dart')
SCENES = os.environ.get('SCENES', 'sandiego,sandiego2').split(',')
RHOS = [float(x) for x in os.environ.get('RHOS', '0.01').split(',')]
SEEDS = [int(x) for x in os.environ.get('SEEDS', '42,43').split(',')]
EPOCHS = int(os.environ.get('EPOCHS', 10000))
THETA = float(os.environ.get('THETA', 0.075))
CKPT_EVERY = int(os.environ.get('CKPT_EVERY', 200))
WD_DARTS = float(os.environ.get('WD', 1e-5))
OUT_JSON = 'results_sd.json'
CKPT_ROOT = 'ckpt_sd'

SP_CFG = load_cfg()
os.makedirs(CKPT_ROOT, exist_ok=True)
_SC = {}


def get_scene(name):
    if name not in _SC:
        sc = scenes.build(name, SP_CFG)
        k = int(SP_CFG['k'])
        img = torch.tensor(np.asarray(sc['te'], np.float32)
                           .reshape(*sc['te_shape'], -1))
        _, nbr = extract_neighborhoods(img, k)
        sc['_te_nbr'] = nbr.numpy()
        img = torch.tensor(np.asarray(sc['tr'], np.float32)
                           .reshape(*sc['tr_shape'], -1))
        _, nbr = extract_neighborhoods(img, k)
        sc['_tr_nbr'] = nbr.numpy()
        _SC[name] = sc
    return _SC[name]


def std_front(tr):
    X = np.asarray(tr, np.float64)
    return Whitening(X.mean(0).astype(np.float32),
                     np.diag(1.0 / X.std(0)).astype(np.float32))


def metrics(y, T):
    return (float(auc_safe(y, T)),
            float(dr_at_fpr(y, T, fpr_list=(0.05,))['0.05']))


def cfar(T, shape, win, guard):
    return DARTS.local_moment_normalize(
        T, shape, win, guard=guard, cfar_lam=float(SP_CFG['cfar_lam']))


def run_one(scene, rho, seed):
    key = f'{DET}_{scene}_r{rho}_s{seed}'
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    if key in res:
        print('skip (done):', key); return
    t0 = time.time()
    sc = get_scene(scene)
    tr, te, s = sc['tr'], sc['te'], np.asarray(sc['sig'], np.float32)
    D = tr.shape[1]
    W = std_front(tr)
    sigma = float(np.sqrt(rho * np.asarray(tr, np.float64).var(0).mean()))
    planted, labels, _ = plant_targets(
        te, s, THETA, float(SP_CFG['target_fraction']), model='additive',
        seed=seed, spatial_shape=sc['te_shape'],
        edge_guard=int(SP_CFG['edge_guard']))
    planted = planted.astype(np.float32)
    y = np.asarray(labels)
    seed_all(seed)
    if DET == 'dart':
        width = int((SP_CFG.get('net_width') or {}).get(scene, 128))
        net = ScoreNet(D, [width], 'relu', whitening=W).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=5e-4, weight_decay=0.0)
        batch = 512
        cwin = int(SP_CFG['dart_cfar_window'])
        cguard = int(SP_CFG['dart_cfar_guard'])
    else:
        c = dict(SP_CFG['darts'])
        net = _NeighborDenoiser(D, int(c['d_lat']), int(c['K']),
                                list(c['enc_hidden']), list(c['score_hidden']),
                                float(np.sqrt(c['dsm_sigma_rho'])),
                                c['activation'], W).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=float(c['lr']),
                                weight_decay=WD_DARTS)
        batch = int(c['batch_size'])
        cwin = int(SP_CFG['darts_cfar_window'] or SP_CFG['k'])
        cguard = int(SP_CFG['darts_cfar_guard'])
    X = torch.tensor(np.asarray(tr, np.float32), device=DEVICE)
    N = (torch.tensor(np.asarray(sc['_tr_nbr'], np.float32), device=DEVICE)
         if DET == 'darts' else None)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(97 * seed)
    P = len(X)
    rundir = os.path.join(CKPT_ROOT, key)
    os.makedirs(rundir, exist_ok=True)

    def score_all(pix, nbr):
        out = []
        with torch.no_grad():
            for i in range(0, len(pix), 1024):
                p = torch.tensor(np.asarray(pix[i:i+1024], np.float32),
                                 device=DEVICE)
                if DET == 'darts':
                    nb = torch.tensor(np.asarray(nbr[i:i+1024], np.float32),
                                      device=DEVICE)
                    out.append(net(p, nb).cpu().numpy())
                else:
                    out.append(net(p).cpu().numpy())
        return np.concatenate(out, 0)

    def detector_T():
        z_tr = score_all(tr, sc['_tr_nbr'])
        z_te = score_all(planted, sc['_te_nbr'])
        zb = z_tr.mean(0)
        C = np.cov(z_tr, rowvar=False)
        return -((z_te - zb) @ s) / np.sqrt(float(s @ C @ s))

    curve = []
    best = {'auc': -1.0}
    best_state = None
    bar = tqdm(range(1, EPOCHS + 1), desc=key, ncols=130, mininterval=5.0,
               file=sys.stdout, ascii=True)
    for ep in bar:
        net.train()
        perm = torch.randperm(P, generator=gen, device=DEVICE)
        for i in range(0, P, batch):
            sel = perm[i:i + batch]
            eps = torch.randn((len(sel), D), generator=gen,
                              device=DEVICE) * sigma
            psi = net(X[sel] + eps, N[sel]) if DET == 'darts' \
                else net(X[sel] + eps)
            loss = ((psi + eps / sigma ** 2) ** 2).sum(-1).mean()
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        if ep % CKPT_EVERY == 0 or ep == EPOCHS:
            net.eval()
            T = detector_T()
            Tc = cfar(T, sc['te_shape'], cwin, cguard)
            auc, pd05 = metrics(y, T)
            auc_c, pd05_c = metrics(y, Tc)
            curve.append({'epoch': ep, 'auc': round(auc, 4),
                          'pd05': round(pd05, 4),
                          'auc_cfar': round(auc_c, 4),
                          'pd05_cfar': round(pd05_c, 4)})
            if auc > best['auc']:
                best = dict(curve[-1]); best['auc'] = auc
                best_state = {k_: v.cpu().clone()
                              for k_, v in net.state_dict().items()}
            torch.save({'w16': {k_: v.half().cpu() for k_, v in
                                net.state_dict().items()}, 'epoch': ep},
                       os.path.join(rundir, f'snap_{ep:06d}.pt'))
            bar.set_postfix_str(f'loss={float(loss):.3g} auc={auc:.3f} '
                                f'cfar={auc_c:.3f} '
                                f'best={best["auc"]:.3f}@{best["epoch"]}')
    bar.close()
    net.eval()
    T = detector_T()
    Tc = cfar(T, sc['te_shape'], cwin, cguard)
    np.savez_compressed(os.path.join(rundir, 'scores_final.npz'),
                        T=T, T_cfar=Tc, labels=y)
    torch.save({'net_final': {k_: v.cpu() for k_, v in
                              net.state_dict().items()},
                'net_best': best_state, 'best': best, 'rho': rho,
                'seed': seed, 'theta': THETA, 'sigma_raw': sigma},
               os.path.join(rundir, 'model.pt'))
    out = {'det': DET, 'scene': scene, 'rho': rho, 'seed': seed,
           'theta': THETA, 'epochs_done': EPOCHS, 'best': best,
           'final': curve[-1], 'curve': curve, 'sigma_raw': round(sigma, 1),
           'cfar_win': cwin, 'sec': round(time.time() - t0)}
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    res[key] = out
    json.dump(res, open(OUT_JSON, 'w'), indent=1)
    print(f'[{key}] best_auc={best["auc"]:.3f}@{best["epoch"]} '
          f'final auc={curve[-1]["auc"]} cfar={curve[-1]["auc_cfar"]} '
          f'({out["sec"]}s)', flush=True)


if __name__ == '__main__':
    done = set(json.load(open(OUT_JSON)).keys()) if os.path.exists(OUT_JSON) else set()
    tasks = [(sc_, r, sd) for sc_ in SCENES for r in RHOS for sd in SEEDS
             if f'{DET}_{sc_}_r{r}_s{sd}' not in done]
    print(f'{len(tasks)} tasks [{DET}], {EPOCHS} ep, device={DEVICE}',
          flush=True)
    for sc_, r, sd in tasks:
        run_one(sc_, r, sd)
    print('ALL DONE', flush=True)

In [ ]:
# ---- DART on SD1+SD2 ----
import os
os.environ['DET'] = 'dart'
os.environ['RHOS'] = DART_RHOS
os.environ['EPOCHS'] = str(DART_EPOCHS)
!python run_sd.py

In [ ]:
# ---- DARTS on SD1+SD2 (plain + CFAR metrics both recorded) ----
import os
os.environ['DET'] = 'darts'
os.environ['RHOS'] = DARTS_RHOS
os.environ['EPOCHS'] = str(DARTS_EPOCHS)
os.environ['WD'] = DARTS_WD
!python run_sd.py

In [ ]:
# ---- Baselines: AMF-global / AMF-local / GMM-Levin / LRao (val-ES) ----
import json, os
import numpy as np, torch
from run_sd import get_scene, SP_CFG, THETA
from repro.core.data import plant_targets
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.protocols.spatial import _windows
from repro.models.classical import AMF, AMFLocal, GMMLevin
from repro.models.lrao import LRao

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS_B = [int(x) for x in os.environ.get('SEEDS', '42,43').split(',')]
SCENES_B = os.environ.get('SCENES', 'sandiego,sandiego2').split(',')

res = {}
for scene in SCENES_B:
    sc = get_scene(scene)
    tr, te, shape = sc['tr'], sc['te'], sc['te_shape']
    sig = np.asarray(sc['sig'], np.float32)
    amf = AMF(SP_CFG).fit(tr)
    lev = GMMLevin(SP_CFG).fit(tr)
    amf_local = AMFLocal(SP_CFG)
    wA = amf_local.resolved_window(tr.shape[1])
    _, nbr_amf = _windows(te, shape, wA, DEVICE)
    lrao_cfg = dict(SP_CFG['lrao'])
    w = (SP_CFG.get('net_width') or {}).get(scene)
    if w: lrao_cfg['hidden'] = [int(w)]
    for seed in SEEDS_B:
        planted, labels, _ = plant_targets(
            te, sig, THETA, float(SP_CFG['target_fraction']),
            model='additive', seed=seed, spatial_shape=shape,
            edge_guard=int(SP_CFG['edge_guard']))
        planted = planted.astype(np.float32); y = np.asarray(labels)
        lrao = LRao(lrao_cfg).fit(tr, seed, DEVICE,
                                  run_dir=f'ckpt_sd/lrao_{scene}_s{seed}')
        scores = {
            'AMF-global': amf.score(planted, sig),
            'AMF-local':  amf_local.score(planted, nbr_amf, sig,
                                          device=DEVICE),
            'GMM-Levin':  lev.score(planted, sig),
            'LRao':       lrao.score(planted, tr[lrao.fit_idx], sig),
        }
        np.savez_compressed(f'ckpt_sd/baseline_scores_{scene}_s{seed}.npz',
                            labels=y, **scores)
        for name, T in scores.items():
            auc = float(auc_safe(y, T))
            pd05 = float(dr_at_fpr(y, T, fpr_list=(0.05,))['0.05'])
            res.setdefault(scene, {}).setdefault(name, {})[f's{seed}'] = \
                {'auc': round(auc, 4), 'pd05': round(pd05, 4)}
            print(f'[{scene} {name} s{seed}] auc={auc:.3f} pd05={pd05:.3f}',
                  flush=True)
for scene, d in res.items():
    for name, dd in d.items():
        dd['auc_mean'] = round(float(np.mean(
            [v['auc'] for k, v in dd.items() if k.startswith('s')])), 4)
json.dump(res, open('results_sd_baselines.json', 'w'), indent=1)
print(json.dumps(res, indent=1))

In [ ]:
# ---- Summary table ----
import json
import numpy as np
r = json.load(open('results_sd.json')) if __import__('os').path.exists('results_sd.json') else {}
rows = {}
for k, v in r.items():
    lab = f"{v['det'].upper()} rho={v['rho']}"
    rows.setdefault((v['scene'], lab), []).append(
        (v['best']['auc'], v['final']['auc'], v['final']['auc_cfar']))
print(f"{'scene':<10} {'model':<18} {'best_auc':>9} {'final':>7} {'final_cfar':>11}")
for (scene, lab), vals in sorted(rows.items()):
    b, f, c = (np.mean([x[i] for x in vals]) for i in range(3))
    print(f'{scene:<10} {lab:<18} {b:>9.3f} {f:>7.3f} {c:>11.3f}')
try:
    bl = json.load(open('results_sd_baselines.json'))
    for scene, d in bl.items():
        for name, dd in d.items():
            print(f'{scene:<10} {name:<18} {dd["auc_mean"]:>9.3f}')
except FileNotFoundError:
    pass

In [ ]:
# ---- Archive EVERYTHING (results + checkpoints + score maps) ----
import shutil, os
!zip -q -r sd_results.zip results_sd.json results_sd_baselines.json ckpt_sd
print(os.path.getsize('sd_results.zip')/1e6, 'MB')
from google.colab import files
shutil.copy('sd_results.zip', 'sd_results_dl.zip')   # copy-then-download
files.download('sd_results_dl.zip')